# 47. Redis — `redis`
**Python Libraries Lab · Phase 3 · Databases**

_An in-memory store for caches, counters, queues and sessions._

⚙️ **Runs in Colab** after the setup cell below.

Redis keeps data in memory, so reads and writes take well under a millisecond. Apps use it for things that must be fast and are often temporary. Caching slow results, counting page views, storing login sessions, rate limiting, and passing jobs between programs through queues.

**In this notebook:** You will store and expire a login code, count product views, keep a shopping cart in a hash, run a simple job queue with a list, build a leaderboard of best sellers, and speed up a slow function with a Redis cache.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Redis is a server. The next cell installs and starts it inside Colab.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q redis
import importlib.metadata as md
for pkg in ['redis']:
    print(pkg, md.version(pkg))

In [ ]:
# Install and start a Redis server inside Colab (about 20 seconds)
!apt-get -qq install -y redis-server > /dev/null 2>&1
!redis-server --daemonize yes --port 6379
import time; time.sleep(1)
!redis-cli ping

## Step 1 · Connect and basic keys
`decode_responses=True` gives normal strings back instead of bytes.

In [ ]:
import redis, time
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
r.flushdb()
print("ping:", r.ping())
r.set("store:name", "ShopEasy")
print(r.get("store:name"))

## Step 2 · Expiring keys (TTL)
A one-time login code that disappears after 2 seconds.

In [ ]:
r.set("otp:asha", "482913", ex=2)
print("code:", r.get("otp:asha"), "| seconds left:", r.ttl("otp:asha"))
time.sleep(2.5)
print("after 2.5s:", r.get("otp:asha"))

## Step 3 · Counters and hashes
`INCR` is atomic, safe even with many servers. A hash is a small dictionary, great for a cart.

In [ ]:
for _ in range(5):
    r.incr("views:product:2")
print("Yoga Mat views:", r.get("views:product:2"))

r.hset("cart:asha", mapping={"steel-bottle": 2, "notebook": 3})
r.hincrby("cart:asha", "notebook", 1)
print("Asha's cart:", r.hgetall("cart:asha"))

## Step 4 · A job queue and a leaderboard
Lists work as queues (push on one end, pop from the other). Sorted sets keep a ranking.

In [ ]:
for job in ["email:1001", "invoice:1001", "email:1002"]:
    r.rpush("jobs", job)
while (job := r.lpop("jobs")):
    print("processing", job)

r.zadd("bestsellers", {"Steel Bottle": 120, "Yoga Mat": 85, "Notebook": 230})
r.zincrby("bestsellers", 50, "Yoga Mat")
print("Top 3:", r.zrevrange("bestsellers", 0, 2, withscores=True))

## Step 5 · Mini project: a cache for a slow function
The first call takes 1.5 s; repeats come from Redis instantly until the key expires.

In [ ]:
import json, functools

def redis_cache(ttl):
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(*args):
            key = f"cache:{fn.__name__}:{json.dumps(args)}"
            if (hit := r.get(key)) is not None:
                return json.loads(hit)
            value = fn(*args)
            r.set(key, json.dumps(value), ex=ttl)
            return value
        return wrapper
    return deco

@redis_cache(ttl=60)
def sales_report(city):
    time.sleep(1.5)                   # pretend this is a slow database query
    return {"city": city, "revenue": len(city) * 10_000}

for _ in range(3):
    t = time.perf_counter(); out = sales_report("Pune")
    print(out, f"{time.perf_counter() - t:.3f}s")

## ✍️ Your turn
Increment the key `visits:home` three times with `r.incr`, then store its value **as an int** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 3, f"Expected 3, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
r.delete("visits:home")
for _ in range(3):
    r.incr("visits:home")
answer = int(r.get("visits:home"))

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Key / value**
- **TTL (expiry)**
- **INCR counter**
- **Hash**
- **List queue**
- **Cache**

**Next:** 48. Testing with pytest (`pytest`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/48_pytest.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install redis
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.